# The Error No Low-Rank Method Can Beat

**Topic:** Low-rank attention &nbsp;·&nbsp; **Difficulty:** Medium

Linformer, Nystromformer, and every other rank-`k` scheme are bounded by the same
thing: the best possible rank-`k` approximation, which Eckart-Young hands you from
the singular values. If that floor is already 90% error, no amount of clever
projection will save the method.

Compute the floor for several attention patterns and predict, before implementing
anything, which heads survive approximation.

## The data (created in the next cell)

- Four attention matrices `(512, 512)`, each row a valid probability
  distribution: diffuse, positional, previous-token, and induction-like.
- `ranks = [8, 16, 32, 64, 128]`.

## Your task

With **NumPy only**:

1. For each attention matrix and each rank, compute the Eckart-Young relative
   Frobenius error floor: `sqrt(sum_{i>k} s_i^2 / sum_i s_i^2)`.
2. `energy_top64` is the fraction of squared singular values in the top 64, per
   head.
3. `rank_for_90` is the smallest rank whose floor is below 0.1, per head
   (`-1` if none of the tested ranks qualify, so search all 512).
4. Set `answers` to a dict with `"floors"` `(4, 5)`, `"energy_top64"` `(4,)`,
   `"rank_for_90"` `(4,)`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-8. The evaluation cell
checks the diffuse head is nearly low-rank while the retrieval-style heads are
not.

In [ ]:
import numpy as np

rng = np.random.default_rng(5004)
n = 512

def rownorm(M):
    M = np.maximum(M, 0)
    return M / M.sum(axis=1, keepdims=True)

idx = np.arange(n)
diffuse = rownorm(np.exp(-((idx[:, None] - idx[None, :]) / 180.0) ** 2))
positional = rownorm(np.exp(-((idx[:, None] - idx[None, :] - 12) / 6.0) ** 2))
prev_token = np.eye(n)[np.clip(idx - 1, 0, None)]
perm = rng.permutation(n)
induction = np.eye(n)[perm]

heads = [diffuse, positional, prev_token, induction]
ranks = [8, 16, 32, 64, 128]
print(f"four {n}x{n} attention patterns: diffuse, positional, previous-token, induction")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("floors", [4, 5], 11.662061232976134, -1.1673478007035412, 1e-08),
    ("energy_top64", [4], 2.234976129253333, 0.022317683383959452, 1e-08),
    ("rank_for_90", [4], 1086.0, 37.5015572922789, 1e-08),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "energy_top64" in answers:
    _e = np.asarray(answers["energy_top64"], dtype=float)
    _f = np.asarray(answers["floors"], dtype=float)
    _diffuse_ok = _e[0] > 0.95
    _retrieval_bad = _e[2] < 0.2 and _e[3] < 0.2
    _floor_high = _f[3, 3] > 0.8
    print(f"  {'diffuse is low rank':<24} {'ok' if _diffuse_ok else 'wrong'}  "
          f"(top-64 energy {_e[0]:.3f})")
    print(f"  {'retrieval heads are not':<24} {'ok' if _retrieval_bad else 'wrong'}  "
          f"({_e[2]:.3f}, {_e[3]:.3f})")
    print(f"  {'rank-64 floor is huge':<24} {'ok' if _floor_high else 'wrong'}  "
          f"({_f[3, 3]:.3f} error, unavoidable)")
    _ok = _ok and _diffuse_ok and _retrieval_bad and _floor_high

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The induction head is a permutation matrix, so it has 512 equal singular values:
its energy is spread perfectly evenly, truncating to rank 64 discards 87.5% of
it, and the floor is `sqrt(1 - 64/512) = 0.935`. The previous-token head is a
shift rather than a permutation, with singular values `sqrt(2)`, `1` repeated 510
times, and `0`, so its rank-64 floor is `sqrt(447/512) = 0.934`, essentially the
same. The diffuse head's Gaussian rows are genuinely
smooth and a handful of components capture them. That single calculation, one SVD
per head, predicts which capabilities a low-rank method will destroy, and it is
the check the 2020-2021 efficient-attention literature mostly skipped.

```python
import numpy as np

floors = np.zeros((len(heads), len(ranks)))
energy = np.zeros(len(heads))
rank90 = np.zeros(len(heads))

for h, M in enumerate(heads):
    s = np.linalg.svd(M, compute_uv=False)
    s2 = s ** 2
    total = s2.sum()
    tail = np.concatenate([np.cumsum(s2[::-1])[::-1][1:], [0.0]])
    for j, k in enumerate(ranks):
        floors[h, j] = float(np.sqrt(tail[k - 1] / total))
    energy[h] = float(s2[:64].sum() / total)
    below = np.flatnonzero(np.sqrt(tail / total) < 0.1)
    rank90[h] = float(below[0] + 1) if len(below) else -1.0

answers = {"floors": floors, "energy_top64": energy, "rank_for_90": rank90}
```